# Step 1 **데이터 이해**

In [ ]:
# Colab setup
%pip install -q mne mne-bids openneuro-py pandas matplotlib mne-icalabel mne-connectivity

In [ ]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd
from IPython.display import display
from mne_bids import BIDSPath, print_dir_tree, read_raw_bids
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
from mne_icalabel import label_components
from mne_connectivity import spectral_connectivity_epochs, spectral_connectivity_time
from mne_connectivity.viz import plot_sensors_connectivity

In [ ]:
# 필요 데이터셋 다운로드

dataset = "ds007526"
target_dir = "/content/openneuro"

subjects = ["sub-003", "sub-004"]

# 공통 메타파일 먼저 다운로드 (한 번만)
!openneuro-py download \
    --dataset={dataset} \
    --target-dir={target_dir} \
    --include=dataset_description.json \
    --include=participants.tsv \
    --include=participants.json \
    --include=README

# 각 subject 다운로드
for sub in subjects:
    print(f"Downloading {sub} ...")
    !openneuro-py download \
        --dataset={dataset} \
        --target-dir={target_dir} \
        --include={sub}/

In [ ]:
# 피험자 정보 확인

df_selected = (
    pd.read_csv(f"{target_dir}/participants.tsv", sep="\t")
    .query("participant_id in @subjects")
    .reset_index(drop=True)
)

df_selected

In [ ]:
bids_root = "/content/openneuro"

bids_path = BIDSPath(
    root = bids_root,
    subject = "003",
    task="rest",
    datatype="eeg",
)

raw = read_raw_bids(bids_path = bids_path, verbose=True)
raw.load_data()

print(raw)

In [ ]:
# raw plot

raw.plot(
    n_channels = 68,
    remove_dc = True
)

In [ ]:
# 채널 이름: 종류

for name, ch_type in zip(raw.ch_names, raw.get_channel_types()):
    print(f"{name}: {ch_type}")

In [ ]:
# 전극 위치 확인

raw.plot_sensors(show_names = True)

In [ ]:
# EOG 채널, VREF 채널 변경

change_channel = raw.copy()

change_channel.set_channel_types({
    "EOG1": "eog",
    "EOG2": "eog",
    "EOG3": "eog",
    "EOG4": "eog",
    "VREF": "misc",
})

change_channel.plot_sensors(show_names=True)

print("EEG:", len(change_channel.copy().pick("eeg").ch_names))
print("EOG:", len(change_channel.copy().pick("eog").ch_names))

# Step 2 **기본 전처리**

In [ ]:
crop = change_channel.copy()

crop.crop(tmin = 20,
          tmax = 200,
          include_tmax = True)

crop

In [ ]:
vref_drop = crop.copy()
vref_drop.drop_channels(["VREF"])

vref_drop

In [ ]:
vref_drop.plot(
    n_channels = 64,
    bad_color = 'red',
    remove_dc = True

)

In [ ]:
ref = vref_drop.copy()

ref.set_eeg_reference(ref_channels = 'average')

ref

In [ ]:
ref.plot(
    n_channels = 64,
    bad_color = 'red',
    remove_dc = True
)

In [ ]:
ref.compute_psd().plot()

In [ ]:
notch = ref.copy()

notch.notch_filter(
    freqs=[50, 100],
    method='fir',
    verbose=True,
    fir_design = 'firwin'
)

In [ ]:
notch.compute_psd().plot()

In [ ]:
bandpass = notch.copy()

bandpass.filter(
    l_freq = 0.5,
    h_freq = 40.0,
    method = 'fir',
    fir_design = 'firwin'
)

In [ ]:
bandpass.plot(
    picks = 'eeg',
    n_channels = 65,
    bad_color = 'red',
    remove_dc = 'True'
)

In [ ]:
bandpass.compute_psd(fmin = 0, fmax = 45).plot()

# Step 3 **인공물 제거**

In [ ]:
raw_ica = notch.copy().filter(l_freq = 1, h_freq = 100, method = 'fir', fir_design = 'firwin')

raw_ica.pick(['eeg', 'eog'])

ica = ICA(
    n_components = 0.999,
    method = 'infomax',
    fit_params = dict(extended = True),
    random_state = 42,
    max_iter = 'auto',
)

ica.fit(raw_ica, picks = 'eeg')

print(ica)

In [ ]:
# EOG 채널을 기준으로 눈깜빡임/안구운동 관련 성분 찾기
eog_inds, eog_scores = ica.find_bads_eog(raw_ica)

print("EOG-related ICA components:", eog_inds)
print("Scores:", eog_scores)

# 일단 후보 지정
ica.exclude = eog_inds

In [ ]:
ica.plot_scores(eog_scores)

In [ ]:
ica.plot_properties(raw_ica, picks=eog_inds, verbose = False)

In [ ]:
ica.plot_components(cmap = 'jet_r')

In [ ]:
ica.plot_sources(inst = raw_ica)

In [ ]:
clean = crop.copy()
ica.apply(clean)

In [ ]:
clean.plot(n_channels = 65,)

# 3-2 2nd ICA

In [ ]:
raw_ica2 = notch.copy().filter(l_freq = 1, h_freq = 100, method = 'fir', fir_design = 'firwin')

ica = ICA(
    n_components = 0.999,
    method = 'infomax',
    fit_params = dict(extended = True),
    random_state = 42,
    max_iter = 'auto',
)

ica.fit(raw_ica2)

print(ica)

In [ ]:
# ICLabel 활용 자동 라벨링

ic_labels = label_components(raw_ica2, ica, method="iclabel")

print(ic_labels.keys())
print(ic_labels["labels"])

In [ ]:
# ICLabel 결과를 보기 좋게 정리

pred_labels = ic_labels["labels"]
pred_proba = np.asarray(ic_labels["y_pred_proba"])

all_ic_info = []


if pred_proba.ndim == 2:
    classes = [
        "brain",
        "muscle artifact",
        "eye blink",
        "heart beat",
        "line noise",
        "channel noise",
        "other",
    ]
    class_to_idx = {c: j for j, c in enumerate(classes)}

    for comp_idx, lab in enumerate(pred_labels):
        row = {"component": comp_idx, "label": lab}
        for j, cls in enumerate(classes):
            row[cls] = float(pred_proba[comp_idx, j])
        all_ic_info.append(row)

else:
    for comp_idx, (lab, prob) in enumerate(zip(pred_labels, pred_proba)):
        all_ic_info.append({
            "component": comp_idx,
            "label": lab,
            "top_probability": float(prob),
        })

for row in all_ic_info:
    print(row)

In [ ]:
# 14. ICA compnent 확인

ica.plot_components(inst = raw_ica2, cmap = 'jet_r')

In [ ]:
# 전체 properties 확인

ica.plot_properties(raw_ica2, picks=range(ica.n_components_), psd_args={"fmin": 0, "fmax": 40}, verbose = False)

In [ ]:
ica.plot_sources(inst = raw_ica2)

In [ ]:
# ICLabel 자동 제거 기준
ICLABEL_EXCLUDE_LABELS = {
    "eye blink",
    "muscle artifact",
    "heart beat",
    "line noise",
    "channel noise",
}

ICLABEL_THRESH = {
    "eye blink": 0.70,
    "muscle artifact": 0.70,
    "heart beat": 0.70,
    "line noise": 0.70,
    "channel noise": 0.70,
}

In [ ]:
# 13. ICLabel 기준 자동 exclude

exclude = []
exclude_detail = []

if pred_proba.ndim == 2:
    classes = [
        "brain",
        "muscle artifact",
        "eye blink",
        "heart beat",
        "line noise",
        "channel noise",
        "other",
    ]
    class_to_idx = {c: j for j, c in enumerate(classes)}

    for comp_idx, lab in enumerate(pred_labels):
        if lab in ICLABEL_EXCLUDE_LABELS:
            prob = float(pred_proba[comp_idx, class_to_idx[lab]])
            if prob >= ICLABEL_THRESH[lab]:
                exclude.append(comp_idx)
                exclude_detail.append(f"{comp_idx}:{lab}:{prob:.2f}")

else:
    for comp_idx, (lab, prob) in enumerate(zip(pred_labels, pred_proba)):
        prob = float(prob)
        if lab in ICLABEL_EXCLUDE_LABELS and prob >= ICLABEL_THRESH[lab]:
            exclude.append(comp_idx)
            exclude_detail.append(f"{comp_idx}:{lab}:{prob:.2f}")

ica.exclude = exclude

print("Excluded ICA components:", ica.exclude)
print("Exclude detail:", exclude_detail)

In [ ]:
post_ica = ica.apply(crop.copy())

post_ica

In [ ]:
post_ica.plot(
    picks = 'eeg',
    n_channels = 64,
    bad_color='red'
)

In [ ]:
post_ica.compute_psd(fmin = 1, fmax = 40, method = 'welch').plot(average=False)

In [ ]:
epochs = mne.make_fixed_length_epochs(post_ica, duration = 5, overlap = 0, preload = True )
epochs

In [ ]:
# EEG 채널만
epochs_eeg = epochs.copy().pick("eeg")


# 주파수 대역 정의
bands = {
    "Delta (0.5-4 Hz)": (0.5, 4),
    "Theta (4-8 Hz)": (4, 8),
    "Alpha (8-13 Hz)": (8, 13),
    "Beta (13-30 Hz)": (13, 30),
}

fig, axes = plt.subplots(1, len(bands), figsize=(12, 4))

for ax, (band_name, (fmin, fmax)) in zip(axes, bands.items()):
    # PSD 계산
    psd = epochs_eeg.compute_psd(
        method="multitaper",
        fmin=fmin,
        fmax=fmax,
        picks="eeg"
    )


    psd_data = psd.get_data()


    band_power = psd_data.mean(axis=0).mean(axis=1)  # (n_channels,)

    mne.viz.plot_topomap(
        band_power,
        epochs_eeg.info,
        cmap = 'jet_r',
        axes=ax,
        contours=6,
        show=False
    )
    ax.set_title(band_name)

plt.tight_layout()
plt.show()